# Bước 4: Data visualization – trực quan hoá dữ liệu

Con số ở bước 2 cho biết *có* mối liên hệ; biểu đồ cho thấy nó *trông ra sao*: đường thẳng, bậc thang, hay đổi chiều.
Các hàm vẽ nằm ở `pipelines/visualization/nodes.py`. Chạy `kedro run --pipeline eda` để lưu tất cả ra `data/08_reporting/figures/eda/`.

In [ ]:
import importlib.util
import logging
import sys
import warnings
from pathlib import Path

# Kiểm tra kernel đã có đủ thư viện chưa, để báo lỗi dễ hiểu nếu thiếu
required = ["kedro", "kedro_datasets", "pandas", "pyarrow", "scipy", "sklearn", "matplotlib"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise ModuleNotFoundError(
        f"Kernel này ({sys.executable}) thiếu: {missing}.\n"
        "Chạy trong terminal rồi khởi động lại kernel:\n"
        f'  "{sys.executable}" -m pip install -r requirements.txt'
    )

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project


def find_project_root() -> Path:
    # Tìm thư mục gốc của dự án Kedro, dù notebook được mở từ đâu
    here = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "x")).resolve().parent
    for folder in (here, *here.parents, Path.cwd(), *Path.cwd().parents):
        if (folder / "conf").is_dir() and (folder / "pyproject.toml").exists():
            return folder
    for pyproject in Path.cwd().glob("*/pyproject.toml"):  # mở từ thư mục cha
        if "[tool.kedro]" in pyproject.read_text(encoding="utf-8"):
            return pyproject.parent
    raise FileNotFoundError("Không tìm thấy dự án Kedro (thư mục có conf/ và pyproject.toml)")


project_path = find_project_root()
bootstrap_project(project_path)  # thêm thư mục src vào đường dẫn import
with KedroSession.create(project_path=project_path) as session:
    context = session.load_context()
    catalog = context.catalog  # dữ liệu khai báo trong conf/base/catalog.yml
    params = context.params    # tham số trong conf/base/parameters*.yml

for name in ("kedro", "netflix"):  # bớt log của Kedro trong notebook
    logging.getLogger(name).setLevel(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("Python đang dùng:", sys.executable)
print("Thư mục dự án:", project_path)

In [ ]:
import io

from IPython.display import Image, display


def show(fig, dpi=110):
    # Hiển thị một Figure trong notebook
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi)
    display(Image(buffer.getvalue()))

In [ ]:
from netflix.pipelines.data_processing.nodes import clean_data
from netflix.pipelines.visualization import nodes as viz

clean = clean_data(catalog.load("netflix_raw"), params["data_processing"])
vp = params["visualization"]
print("Dữ liệu sạch:", clean.shape)

## 1. Hai lớp có cân bằng không?
Cột cao bằng nhau: không cần xử lý mất cân bằng, và accuracy là chỉ số dùng được.

In [ ]:
show(viz.plot_target_balance(clean, vp))

## 2. Phân phối từng cột số, tách theo lớp
Hai màu **chồng lên nhau** nhiều: cột đó khó phân biệt hai lớp. Hai màu **tách xa nhau**: cột đó có ích.
Tìm hai cột mà màu đỏ và xanh tách rõ nhất (gợi ý: các cột về xem phim và đăng nhập).

In [ ]:
show(viz.plot_numeric_distributions(clean, vp))

## 3. Tỉ lệ rời bỏ theo từng nhóm của cột số
Đây là biểu đồ quan trọng nhất của bước 4. Đường nét đứt là tỉ lệ rời chung.
Nhìn `last_login_days` (tăng vọt giữa chừng) và `number_of_profiles` (rơi xuống từ 4 hồ sơ): đây là lý do bước 3 tạo `is_inactive` và `has_many_profiles`.

In [ ]:
show(viz.plot_churn_rate_numeric(clean, vp))

## 4. Tỉ lệ rời bỏ theo từng nhóm của cột chữ
Cột nào các cột cao bằng nhau (gần đường nét đứt) là cột không liên quan: `gender`, `region`, `device`, `favorite_genre`.
Hai cột có chênh lệch rõ là `subscription_type` (Basic rời nhiều nhất) và `payment_method`.

In [ ]:
show(viz.plot_churn_rate_categorical(clean, vp))

## 5. Tương quan giữa các cột số
Hàng/cột `churned` cho biết cột nào liên quan đến việc rời bỏ. Các ô đậm giữa hai **feature** cho biết hai cột đó mang thông tin chồng nhau (ví dụ `watch_hours` và `avg_watch_time_per_day`).

In [ ]:
show(viz.plot_correlation_heatmap(clean, vp))

## Tự thử
Hàm `viz.plot_churn_rate_numeric` chia mỗi cột thành 5 nhóm. Đổi `numeric_bins` trong `conf/base/parameters_visualization.yml` thành 10 rồi chạy lại ô số 3: đường cong có rõ hơn không?

## Ghi chú của bạn
Viết vài dòng bằng lời của bạn: bạn học được gì, điều gì bất ngờ, bạn sẽ thử gì tiếp?

-
-
-